# SLIIT Lecture Summarizer - Training Pipeline

**Approach:** Train a RandomForest classifier on SLIIT lecture PDFs to identify important sentences.

- **No external datasets** (no Kaggle, no arXiv)
- **Smart structural labeling** based on sentence patterns found in lecture content
- **TF-IDF vocabulary built from YOUR lectures** so it generalizes to any SLIIT module

---

## What This Notebook Does

1. Install dependencies
2. Mount Google Drive
3. Upload SLIIT lecture PDFs
4. Extract and clean sentences from PDFs
5. Label sentences using structural pattern matching
6. Train RandomForest model
7. Evaluate with human-verifiable test cases
8. Download model for local deployment

**Time:** ~10-15 minutes total

## STEP 1: Install Dependencies

In [ ]:
import subprocess, sys

packages = ['scikit-learn', 'nltk', 'pandas', 'numpy', 'PyPDF2', 'python-pptx']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print('All dependencies installed.')

## STEP 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

WORK_DIR = '/content/drive/My Drive/SLIIT_Summarizer'
os.makedirs(f'{WORK_DIR}/training_data', exist_ok=True)
os.makedirs(f'{WORK_DIR}/models', exist_ok=True)
os.makedirs(f'{WORK_DIR}/lectures', exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

## STEP 3: Upload Lecture PDFs

Upload **all** your SLIIT lecture PDFs. The more modules you upload, the better the model generalizes.

Minimum: 3-4 PDFs. Recommended: 6+ from different modules.

In [ ]:
from google.colab import files
import shutil

print('Upload your SLIIT lecture PDFs (select multiple files at once)...')
uploaded = files.upload()

for fname in uploaded.keys():
    shutil.move(fname, f'lectures/{fname}')
    print(f'  Saved: lectures/{fname}')

print(f'\nTotal files: {len(uploaded)}')

## STEP 4: Extract & Clean Sentences

In [ ]:
import re
import pandas as pd
from PyPDF2 import PdfReader
from nltk.tokenize import sent_tokenize

def extract_text_from_pdf(path):
    try:
        reader = PdfReader(path)
        text = ''
        for page in reader.pages:
            t = page.extract_text()
            if t:
                text += t + ' '
        return text
    except Exception as e:
        print(f'  Error reading {path}: {e}')
        return ''

def clean_text(text):
    """Remove PDF artifacts, slide headers, and noise."""
    text = re.sub(r'/g\d+', '', text)
    text = re.sub(r'\(cid:\d+\)', '', text)
    text = re.sub(r'\x0c', ' ', text)
    text = re.sub(r'[\x00-\x08\x0b\x0e-\x1f]', '', text)
    text = re.sub(r'\xc2\xa7|§', ' ', text)
    # Remove common slide header patterns
    text = re.sub(r'[A-Z]{2}\d{3,4}\s*\|[^.!?\n]*', ' ', text)
    text = re.sub(r'Module\s+Code\s*\|[^\n]*', ' ', text)
    text = re.sub(r'Faculty\s*of\s*Computing[^.]*', ' ', text)
    text = re.sub(r'Department\s+of\s+\w+[^.]*', ' ', text)
    # Fix camelCase from slide extraction
    text = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', text)
    text = re.sub(r'\n{2,}', '. ', text)
    text = re.sub(r'(?<=[a-z])\n(?=[a-z])', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_clean_sentences(text):
    """Tokenize and filter sentences."""
    sentences = sent_tokenize(text)
    cleaned = []
    for s in sentences:
        s = s.strip()
        # Remove remaining header fragments
        s = re.sub(r'^[A-Z]{2}\d{3,4}[^.]*?(Lecture|Module)\s*\d*', '', s).strip()
        s = re.sub(r'^\d+\s+', '', s).strip()
        s = re.sub(r'\s+\d+\s*$', '', s).strip()
        words = s.split()
        if len(words) < 5 or len(words) > 100:
            continue
        # Must be mostly alphabetic
        alpha_ratio = sum(1 for c in s if c.isalpha()) / max(len(s), 1)
        if alpha_ratio < 0.5:
            continue
        # Skip jammed words (bad PDF extraction)
        long_words = [w for w in words if len(w) > 25 and not w.startswith('http')]
        if len(long_words) > 1:
            continue
        cleaned.append(s)
    return cleaned

# Process all PDFs
all_sentences = []
lecture_files = [f for f in os.listdir('lectures') if f.endswith('.pdf')]

print(f'Processing {len(lecture_files)} lecture PDFs...\n')

for fname in sorted(lecture_files):
    path = f'lectures/{fname}'
    raw_text = extract_text_from_pdf(path)
    cleaned_text = clean_text(raw_text)
    sentences = extract_clean_sentences(cleaned_text)

    for i, s in enumerate(sentences):
        all_sentences.append({
            'sentence': s,
            'source_file': fname,
            'position_ratio': i / max(len(sentences) - 1, 1),
            'word_count': len(s.split())
        })
    print(f'  {fname}: {len(sentences)} sentences')

df = pd.DataFrame(all_sentences)
print(f'\nTotal sentences extracted: {len(df)}')
print(f'Average words per sentence: {df["word_count"].mean():.1f}')

# Save raw extracted sentences
df.to_csv('training_data/extracted_sentences.csv', index=False)
print(f'Saved to: training_data/extracted_sentences.csv')

## STEP 5: Label Sentences

This is the key step. Instead of a crude keyword counter, we use **structural pattern matching** that detects:

**Important (label=1):**
- Definitions: "X is defined as...", "X refers to...", "X is a..."
- Key concepts: sentences with technical terms, acronyms explained
- Formulas/rules: "The formula for...", normalized forms, constraints
- Enumerations: "There are N types of...", "The steps are..."
- Conclusions: "Therefore...", "In summary...", "The key point is..."

**Not important (label=0):**
- Slide headers/titles: "LECTURE 01 - INTRODUCTION"
- Instructor references: "Thank you!", email addresses
- Filler transitions: "As we discussed...", "Moving on..."
- Page/slide artifacts: "Slide 15 of 42", copyright notices
- Short fragments that lack substance

In [ ]:
import re
import numpy as np

def label_sentence(sentence):
    """
    Label a sentence as important (1) or not (0) using structural patterns.
    Returns (label, confidence, reason).

    The rules are designed to generalize across ANY lecture module,
    not just the ones uploaded for training.
    """
    s = sentence.strip()
    sl = s.lower()
    words = s.split()
    word_count = len(words)

    # ========================
    # HARD NEGATIVES (label=0)
    # ========================

    # Slide headers and titles (ALL CAPS, short)
    upper_ratio = sum(1 for c in s if c.isupper()) / max(sum(1 for c in s if c.isalpha()), 1)
    if upper_ratio > 0.7 and word_count < 12:
        return (0, 0.95, 'slide_header')

    # Instructor names, emails, acknowledgments
    if re.search(r'@\w+\.\w+', s):
        return (0, 0.95, 'email')
    if re.search(r'thank\s*you|acknowledgment|references?\s*$', sl):
        return (0, 0.90, 'closing')

    # Copyright, page numbers, slide markers
    if re.search(r'copyright|all rights reserved|page \d+|slide \d+', sl):
        return (0, 0.95, 'boilerplate')

    # Very short filler (< 7 words with no technical content)
    if word_count < 7:
        has_technical = bool(re.search(r'[A-Z]{2,}|\d+[A-Z]|[a-z]+_[a-z]+', s))
        if not has_technical:
            return (0, 0.80, 'too_short')

    # Pure references: "See chapter 5", "Refer to textbook"
    if re.match(r'^(see|refer to|read|check|go through|look at)\s', sl):
        return (0, 0.85, 'reference')

    # Transition filler
    filler_starts = [
        r'^as (we |mentioned|discussed|noted|seen)',
        r'^(moving on|let us|let\'s|now we|we will now)',
        r'^(in the (next|previous|last) (lecture|slide|section))',
        r'^(you (already|should|can|will) (know|have|see))',
        r'^(note that this is just|this is just)',
    ]
    for pattern in filler_starts:
        if re.match(pattern, sl):
            return (0, 0.80, 'filler_transition')

    # Activity/homework prompts (not content)
    if re.match(r'^(activity|exercise|homework|assignment|question)\s', sl):
        return (0, 0.75, 'activity_prompt')

    # ========================
    # HARD POSITIVES (label=1)
    # ========================

    # Definitions: "X is/are [a/an/the] ..."
    if re.match(r'^[A-Z].*?\b(is|are|refers to|is defined as|is known as|means|can be defined as)\b', s):
        if word_count >= 8:
            return (1, 0.90, 'definition')

    # Definitions with dash: "DBMS - A database management system..."
    if re.match(r'^[A-Z].*?\s[-–:]\s', s) and word_count >= 8:
        return (1, 0.85, 'definition_dash')

    # Enumeration of types/categories: "There are N types..."
    if re.search(r'there are (\d+|two|three|four|five|six|seven|several|many|various)\s+(types?|kinds?|categories|levels?|phases?|stages?|steps?|forms?|methods?|ways?|classes?|modes?)', sl):
        return (1, 0.90, 'enumeration')

    # "X consists of / includes / contains"
    if re.search(r'\b(consists? of|is composed of|includes?|comprises?)\b', sl) and word_count >= 8:
        return (1, 0.85, 'composition')

    # Formulas and rules with technical notation
    if re.search(r'\b(formula|equation|theorem|rule|law|axiom|constraint|condition)\b', sl) and word_count >= 7:
        return (1, 0.85, 'formula_rule')

    # Normal forms, protocols, standards (domain-specific importance)
    if re.search(r'\b(\d+NF|BCNF|normal form|ACID|CAP theorem|OSI|TCP|UDP|HTTP|DNS|DHCP|SNMP)\b', s) and word_count >= 7:
        return (1, 0.85, 'standard_protocol')

    # Process descriptions: "The process/procedure/steps..."
    if re.search(r'\b(the (process|procedure|algorithm|method|technique|approach|mechanism) (of|for|to|is|involves))\b', sl):
        return (1, 0.85, 'process_desc')

    # Advantages/disadvantages lists
    if re.search(r'\b(advantages?|disadvantages?|benefits?|drawbacks?|pros?|cons?|strengths?|weaknesses?|limitations?)\b', sl):
        if word_count >= 7:
            return (1, 0.80, 'advantage_disadvantage')

    # Conclusions and key takeaways
    if re.match(r'^(therefore|thus|hence|in summary|in conclusion|the key point|to summarize|consequently)', sl):
        return (1, 0.85, 'conclusion')

    # Comparison statements
    if re.search(r'\b(difference between|compared to|in contrast|unlike|whereas|while.*differs?)\b', sl) and word_count >= 8:
        return (1, 0.80, 'comparison')

    # Sentences with acronym explanations: "SQL (Structured Query Language)"
    if re.search(r'[A-Z]{2,}\s*\([A-Z][a-z]', s) and word_count >= 6:
        return (1, 0.85, 'acronym_definition')

    # "X is used to/for...", "X allows...", "X enables...", "X provides..."
    if re.search(r'\b(is used (to|for)|allows|enables|provides|ensures|guarantees|prevents|supports)\b', sl) and word_count >= 8:
        return (1, 0.80, 'functional_desc')

    # ========================
    # SOFT SIGNALS (scoring)
    # ========================
    score = 0.0

    # Technical depth indicators
    acronyms = len(re.findall(r'\b[A-Z]{2,}\b', s))
    if acronyms >= 2:
        score += 0.20

    # Longer, well-structured sentences (10-50 words) tend to carry content
    if 10 <= word_count <= 50:
        score += 0.15
    elif word_count > 50:
        score -= 0.05

    # Technical vocabulary signals
    tech_terms = len(re.findall(r'\b(data|system|network|protocol|server|client|query|table|process|thread|memory|file|algorithm|function|class|object|module|interface|database|schema|index|transaction|security|architecture|layer|node|packet|buffer|cache|stack|queue|tree|graph)\b', sl))
    score += min(tech_terms * 0.05, 0.25)

    # Has a verb + object structure (actual explanation, not fragment)
    if re.search(r'\b(is|are|was|were|has|have|can|will|must|should|provides?|requires?|allows?|ensures?|creates?|stores?|manages?)\b', sl):
        score += 0.10

    # Causal/explanatory connectors
    if re.search(r'\b(because|since|due to|in order to|so that|as a result|leads to|causes)\b', sl):
        score += 0.15

    # Examples (often illustrate important concepts)
    if re.search(r'\b(for example|e\.g\.|such as|for instance)\b', sl):
        score += 0.10

    # Negative signals
    if re.search(r'\b(obviously|clearly|of course|simply|just|basically)\b', sl):
        score -= 0.10

    # Threshold for soft labels
    if score >= 0.40:
        return (1, 0.60 + min(score, 0.30), 'soft_positive')
    elif score <= 0.05:
        return (0, 0.65, 'soft_negative')

    # Ambiguous - default to not important (conservative)
    return (0, 0.55, 'ambiguous')


# Label all sentences
print('Labeling sentences...\n')

labels = []
confidences = []
reasons = []

for _, row in df.iterrows():
    label, conf, reason = label_sentence(row['sentence'])
    labels.append(label)
    confidences.append(conf)
    reasons.append(reason)

df['importance'] = labels
df['label_confidence'] = confidences
df['label_reason'] = reasons

# Show statistics
n_pos = sum(labels)
n_neg = len(labels) - n_pos
print(f'Total sentences: {len(df)}')
print(f'Important (1):     {n_pos} ({n_pos/len(df)*100:.1f}%)')
print(f'Not important (0): {n_neg} ({n_neg/len(df)*100:.1f}%)')
print(f'\nLabel reasons breakdown:')
print(df['label_reason'].value_counts().to_string())

# Save labeled data
df.to_csv('training_data/labeled_sentences.csv', index=False)
print(f'\nSaved to: training_data/labeled_sentences.csv')

### STEP 5b: Verify Labels (sanity check)

Review samples from each label category to make sure the labeling is correct.

In [ ]:
print('=== SAMPLE IMPORTANT SENTENCES (label=1) ===\n')
pos_df = df[df['importance'] == 1].sample(min(10, n_pos), random_state=42)
for _, row in pos_df.iterrows():
    print(f'  [{row["label_reason"]:20s}] {row["sentence"][:120]}')

print(f'\n=== SAMPLE NOT-IMPORTANT SENTENCES (label=0) ===\n')
neg_df = df[df['importance'] == 0].sample(min(10, n_neg), random_state=42)
for _, row in neg_df.iterrows():
    print(f'  [{row["label_reason"]:20s}] {row["sentence"][:120]}')

print(f'\n=== HIGH-CONFIDENCE LABELS ===\n')
high_conf = df[df['label_confidence'] >= 0.85]
print(f'High-confidence labels (>=0.85): {len(high_conf)} ({len(high_conf)/len(df)*100:.1f}%)')
print(f'  Important: {len(high_conf[high_conf["importance"]==1])}')
print(f'  Not important: {len(high_conf[high_conf["importance"]==0])}')

## STEP 6: Train the Model

Features:
- **TF-IDF** (200 terms from lecture vocabulary, bigrams)
- **Structural features**: word count, avg word length, capitalization, digit ratio, punctuation
- **Pattern features**: has definition pattern, has acronym, has technical terms, sentence position

In [ ]:
import numpy as np
import pickle
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Load labeled data
df = pd.read_csv('training_data/labeled_sentences.csv')
df = df.dropna(subset=['sentence'])
df['sentence'] = df['sentence'].astype(str).str.strip()
df = df[df['sentence'].str.len() > 10]
print(f'Training data: {len(df)} sentences')
print(f'  Class 1 (important): {df["importance"].sum()}')
print(f'  Class 0 (not important): {len(df) - df["importance"].sum()}')

# ========================
# Feature extraction
# ========================

def extract_structural_features(sentence, position_ratio=0.5):
    """Extract non-TF-IDF features that generalize across modules."""
    words = sentence.split()
    word_count = len(words)
    s = sentence
    sl = sentence.lower()

    features = [
        # Basic text stats
        word_count,
        np.mean([len(w) for w in words]) if words else 0,  # avg word length
        sum(1 for c in s if c.isupper()) / max(len(s), 1),  # capital ratio
        sum(1 for c in s if c.isdigit()) / max(len(s), 1),  # digit ratio
        sum(1 for c in s if c in '.,;:!?') / max(len(s), 1),  # punct ratio

        # Structural patterns (binary)
        1 if re.match(r'^[A-Z].*?\b(is|are|refers to|is defined as|means)\b', s) else 0,  # definition
        1 if re.search(r'[A-Z]{2,}\s*\([A-Z]', s) else 0,  # acronym explanation
        1 if re.search(r'\b(consists? of|includes?|comprises?)\b', sl) else 0,  # composition
        1 if re.search(r'there are (\d+|two|three|four|five|six|several)', sl) else 0,  # enumeration
        1 if re.search(r'\b(therefore|thus|hence|in summary|in conclusion)\b', sl) else 0,  # conclusion
        1 if re.search(r'\b(advantage|disadvantage|benefit|drawback|limitation)\b', sl) else 0,  # pros/cons
        1 if re.search(r'\b(because|since|due to|in order to|so that)\b', sl) else 0,  # causal
        1 if re.search(r'\b(for example|e\.g\.|such as|for instance)\b', sl) else 0,  # example
        1 if re.search(r'\b(difference between|compared to|in contrast|unlike)\b', sl) else 0,  # comparison
        1 if re.search(r'\b(is used (to|for)|allows|enables|provides|ensures)\b', sl) else 0,  # functional

        # Technical density
        len(re.findall(r'\b[A-Z]{2,}\b', s)),  # acronym count
        len(re.findall(r'\b(data|system|network|protocol|server|query|table|process|memory|file|algorithm|function|class|database|schema|index|transaction|security|architecture)\b', sl)),  # tech term count

        # Position in document (earlier = more likely introductory/definitional)
        position_ratio,

        # Negative signals
        1 if re.search(r'@\w+\.\w+', s) else 0,  # has email
        1 if re.search(r'copyright|all rights reserved', sl) else 0,  # boilerplate
        1 if re.match(r'^(see|refer to|read|check)\s', sl) else 0,  # reference
    ]
    return features

STRUCTURAL_FEATURE_NAMES = [
    'word_count', 'avg_word_len', 'capital_ratio', 'digit_ratio', 'punct_ratio',
    'is_definition', 'has_acronym_def', 'is_composition', 'is_enumeration',
    'is_conclusion', 'is_pros_cons', 'is_causal', 'is_example', 'is_comparison',
    'is_functional', 'acronym_count', 'tech_term_count', 'position_ratio',
    'has_email', 'is_boilerplate', 'is_reference'
]

# Build TF-IDF from lecture vocabulary
print('\nBuilding TF-IDF vocabulary from lecture content...')
vectorizer = TfidfVectorizer(
    max_features=200,
    min_df=3,
    max_df=0.85,
    ngram_range=(1, 2),
    stop_words='english'
)

tfidf_matrix = vectorizer.fit_transform(df['sentence'].values).toarray()
print(f'TF-IDF vocabulary: {len(vectorizer.vocabulary_)} terms')
print(f'Sample terms: {list(vectorizer.vocabulary_.keys())[:20]}')

# Extract structural features
print('\nExtracting structural features...')
structural_features = np.array([
    extract_structural_features(row['sentence'], row.get('position_ratio', 0.5))
    for _, row in df.iterrows()
])

# Combine: TF-IDF + structural
X = np.hstack([tfidf_matrix, structural_features])
y = df['importance'].values

print(f'\nFinal feature matrix: {X.shape}')
print(f'  TF-IDF features: {tfidf_matrix.shape[1]}')
print(f'  Structural features: {structural_features.shape[1]}')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {len(X_train)}, Test: {len(X_test)}')

# Train RandomForest
print('\nTraining RandomForest...')
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print('Training complete.')

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f'\n{"=" * 50}')
print(f'MODEL PERFORMANCE')
print(f'{"=" * 50}')
print(f'Accuracy:  {accuracy:.4f} ({accuracy*100:.1f}%)')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1 Score:  {f1:.4f}')

print(f'\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Important', 'Important']))

print(f'Confusion Matrix:')
cm = confusion_matrix(y_test, y_pred)
print(f'  TN={cm[0][0]}  FP={cm[0][1]}')
print(f'  FN={cm[1][0]}  TP={cm[1][1]}')

# Cross-validation
print(f'\n5-Fold Cross-Validation...')
cv_scores = cross_val_score(model, X, y, cv=5, scoring='f1')
print(f'  CV F1 scores: {[f"{s:.3f}" for s in cv_scores]}')
print(f'  Mean: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})')

### STEP 6b: Feature Importance Analysis

In [ ]:
# Feature importance analysis
importances = model.feature_importances_
n_tfidf = tfidf_matrix.shape[1]

tfidf_importance = importances[:n_tfidf].sum()
structural_importance = importances[n_tfidf:].sum()

print(f'Feature importance distribution:')
print(f'  TF-IDF features:    {tfidf_importance:.3f} ({tfidf_importance*100:.1f}%)')
print(f'  Structural features: {structural_importance:.3f} ({structural_importance*100:.1f}%)')

print(f'\nTop 10 structural features:')
struct_importances = importances[n_tfidf:]
struct_sorted = sorted(zip(STRUCTURAL_FEATURE_NAMES, struct_importances), key=lambda x: x[1], reverse=True)
for name, imp in struct_sorted[:10]:
    print(f'  {name:25s} {imp:.4f}')

print(f'\nTop 10 TF-IDF terms:')
tfidf_importances = importances[:n_tfidf]
vocab_items = sorted(vectorizer.vocabulary_.items(), key=lambda x: x[1])
tfidf_sorted = sorted(zip([v[0] for v in vocab_items], tfidf_importances), key=lambda x: x[1], reverse=True)
for term, imp in tfidf_sorted[:10]:
    print(f'  {term:25s} {imp:.4f}')

## STEP 7: Test with Real Sentences

Verify the model produces sensible predictions on known examples.

In [ ]:
test_cases = [
    # Should be IMPORTANT
    ('A database is a collection of related data organized for efficient access and retrieval.', 1),
    ('Normalization is the process of organizing data to reduce redundancy and improve integrity.', 1),
    ('The primary key uniquely identifies each record in a relational database table.', 1),
    ('SQL (Structured Query Language) is a standard language for managing relational databases.', 1),
    ('There are three levels of database architecture: external, conceptual, and internal.', 1),
    ('ACID properties include Atomicity, Consistency, Isolation, and Durability.', 1),
    ('A foreign key is a field that refers to the primary key of another table.', 1),
    ('The process scheduling algorithm determines which process runs next on the CPU.', 1),
    ('TCP provides reliable, ordered delivery of data between applications.', 1),
    ('Deadlock occurs when two or more processes are waiting for each other to release resources.', 1),

    # Should be NOT IMPORTANT
    ('Thank you for attending this lecture.', 0),
    ('See chapter 5 for more details on this topic.', 0),
    ('As we discussed in the previous lecture, we will continue.', 0),
    ('Slide 15 of 42 page break continued.', 0),
    ('Copyright 2024 All Rights Reserved SLIIT.', 0),
    ('Moving on to the next section of our discussion.', 0),
    ('Please refer to the textbook for additional reading.', 0),
    ('You already know these concepts from the previous module.', 0),
    ('Any questions so far about what we covered?', 0),
    ('Let us now look at another example.', 0),
]

print(f'{"=" * 60}')
print('HUMAN-VERIFIABLE TEST CASES')
print(f'{"=" * 60}\n')

correct = 0
total = len(test_cases)

for sentence, expected in test_cases:
    tfidf_feat = vectorizer.transform([sentence]).toarray()
    struct_feat = np.array([extract_structural_features(sentence, 0.5)])
    features = np.hstack([tfidf_feat, struct_feat])

    pred = model.predict(features)[0]
    prob = model.predict_proba(features)[0][1]

    match = 'OK' if pred == expected else 'WRONG'
    if pred == expected:
        correct += 1

    label = 'IMP' if pred == 1 else 'NOT'
    exp_label = 'IMP' if expected == 1 else 'NOT'
    print(f'  [{match:5s}] pred={label} exp={exp_label} prob={prob:.2f} | {sentence[:75]}')

print(f'\nTest accuracy: {correct}/{total} ({correct/total*100:.0f}%)')

## STEP 8: Save & Download Model

In [ ]:
import pickle
import json

# Save model
with open('models/universal_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Saved: models/universal_model.pkl')

# Save vectorizer
with open('models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
print('Saved: models/vectorizer.pkl')

# Save metadata
metadata = {
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1': float(f1),
    'cv_f1_mean': float(cv_scores.mean()),
    'cv_f1_std': float(cv_scores.std()),
    'training_approach': 'RandomForest on SLIIT lecture PDFs with structural labeling',
    'training_data': f'{len(df)} sentences from {len(df["source_file"].unique())} lecture PDFs',
    'n_features': int(X.shape[1]),
    'n_tfidf_features': int(tfidf_matrix.shape[1]),
    'n_structural_features': int(structural_features.shape[1]),
    'class_distribution': {
        'important': int(df['importance'].sum()),
        'not_important': int(len(df) - df['importance'].sum())
    }
}

with open('models/universal_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved: models/universal_metadata.json')

print(f'\n{json.dumps(metadata, indent=2)}')

In [ ]:
from google.colab import files

print('Downloading model files...\n')
print('Copy these 3 files to your local project backend/ folder:')
print('  1. universal_model.pkl')
print('  2. vectorizer.pkl')
print('  3. universal_metadata.json')
print()

files.download('models/universal_model.pkl')
files.download('models/vectorizer.pkl')
files.download('models/universal_metadata.json')

print('\nDone! Files also saved in Google Drive: SLIIT_Summarizer/models/')